In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'


In [2]:
import shutil
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import hashlib
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
LOCAL_DATA_ROOT = WORKDIR / 'snuaichallenge_data'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'snuaichallenge_data'

# scripts/build_qwen3vl_v2_os_bundle.py 출력값. 번들을 다시 만들면 갱신할 것.
EXPECTED_BUNDLE_SHA256 = '1240F4E4A6C0DF1D3ACA6B031A6CEF25CABE4731E0F244E1AFC93E80B8156F39'

def has_train_data(root):
    return (root / 'train.csv').is_file() and (root / 'train').is_dir()

def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None

WORKDIR.mkdir(parents=True, exist_ok=True)
if has_train_data(DRIVE_DATA_ROOT):
    DATA_ROOT = DRIVE_DATA_ROOT
else:
    DATA_ROOT = LOCAL_DATA_ROOT
    if not has_train_data(DATA_ROOT):
        train_bundle = find_drive_file('snu_colab_full_train_bundle.tar')
        if train_bundle is None:
            raise FileNotFoundError(
                "MyDrive에서 snu_colab_full_train_bundle.tar를 찾지 못했습니다. "
                "MyDrive/snu ai/에 업로드해 주세요."
            )
        print(f'학습 데이터 압축 해제: {train_bundle}')
        subprocess.run(['tar', '-xf', str(train_bundle), '-C', str(WORKDIR)], check=True)
        DRIVE_ROOT = train_bundle.parent
assert has_train_data(DATA_ROOT), f'학습 데이터가 없습니다: {DATA_ROOT}'

bundle_path = find_drive_file('snu_colab_qwen3vl_v2_os_bundle.tar')
if bundle_path is None:
    raise FileNotFoundError(
        "MyDrive에서 snu_colab_qwen3vl_v2_os_bundle.tar를 찾지 못했습니다. "
        "scripts/build_qwen3vl_v2_os_bundle.py로 만들어 MyDrive/snu ai/에 업로드해 주세요."
    )
bundle_sha = hashlib.sha256(bundle_path.read_bytes()).hexdigest().upper()
assert bundle_sha == EXPECTED_BUNDLE_SHA256, f'번들 sha256 불일치: {bundle_sha}'
subprocess.run(['tar', '-xf', str(bundle_path), '-C', str(WORKDIR)], check=True)

DATA_DIR = str(DATA_ROOT)
OUTPUT_DIR = str(DRIVE_ROOT / 'snu_qwen3vl_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

FOLD_ASSIGNMENTS_CSV = WORKDIR / 'outputs/lora_fold_split/fold_assignments.csv'
SPLIT_METADATA_JSON = WORKDIR / 'outputs/lora_fold_split/split_metadata.json'
OVERSAMPLE_IDS_CSV = WORKDIR / 'outputs/qwen3vl_v2_os_selector/oversample_ids.csv'
SELECTOR_METADATA_JSON = WORKDIR / 'outputs/qwen3vl_v2_os_selector/metadata.json'
for path in (FOLD_ASSIGNMENTS_CSV, SPLIT_METADATA_JSON,
             OVERSAMPLE_IDS_CSV, SELECTOR_METADATA_JSON):
    assert path.is_file(), f'번들에서 풀리지 않은 파일: {path}'

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


학습 데이터 압축 해제: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
DATA_DIR: /content/snu_ai/snuaichallenge_data
OUTPUT_DIR: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs


In [5]:
# 원본 학습 노트북은 비핀 설치였다. 기준선 adapter의 holdout EM 0.5298을
# 재현한 hybrid_judgment_colab.ipynb의 검증된 조합으로 핀한다 (유일한 의도적
# 인프라 변경). Colab 기본 torchao 0.10은 peft LoRA 주입을 깨뜨리므로 함께 올린다.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
    'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0',
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', 'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46', 'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0'], returncode=0)

In [6]:
import os, re, ast, random, hashlib, time, json, math
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, PeftModel, get_peft_model
from qwen_vl_utils import process_vision_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
ARM_NAME = 'qwen3vl_v2_os'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
print(f'train rows: {len(train)}')


train rows: 9535


In [7]:
# 원본의 인라인 fold 재계산(sklearn 버전에 민감)을 대체:
# 번들로 배송된 fold 배정을 그대로 읽는다. 레시피의 분할 자체는 동일하다.
fold_df = pd.read_csv(FOLD_ASSIGNMENTS_CSV)
train = train.merge(fold_df[['Id', 'Fold']], on='Id', how='left', validate='one_to_one')
assert train['Fold'].notna().all(), 'fold 배정이 없는 샘플이 있습니다'
train['fold'] = train['Fold'].astype(int)

fold_sizes = train['fold'].value_counts().to_dict()
assert fold_sizes == {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}, fold_sizes

EXPECTED_HOLDOUT_SHA256 = (
    '82d69f13a10d3f1903694201d020c85d8ca7d71c1c07ae17d2ef80a0bd2f44d0'
)
fold0_ids = sorted(train.loc[train['fold'] == 0, 'Id'])
holdout_sha = hashlib.sha256('\n'.join(fold0_ids).encode('utf-8')).hexdigest()
assert holdout_sha == EXPECTED_HOLDOUT_SHA256, holdout_sha

oversample_df = pd.read_csv(OVERSAMPLE_IDS_CSV)
selector_meta = json.loads(SELECTOR_METADATA_JSON.read_text())
assert selector_meta['threshold'] == 0.9 and selector_meta['multiplier'] == 2
OVERSAMPLE_IDS = set(oversample_df['Id'])
assert len(OVERSAMPLE_IDS) == len(oversample_df) == selector_meta['n_selected']
fold_by_id = dict(zip(train['Id'], train['fold']))
assert all(fold_by_id[sample_id] != 0 for sample_id in OVERSAMPLE_IDS), \
    'fold-0 샘플이 오버샘플 목록에 섞였습니다'
print(f"oversample ids: {len(OVERSAMPLE_IDS)} "
      f"(GT-인접 max cosine > {selector_meta['threshold']}, x{selector_meta['multiplier']})")


oversample ids: 3194 (GT-인접 max cosine > 0.9, x2)


In [8]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses


In [9]:
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)

def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def format_training_sample(row, img_dir):
    messages = get_prompt_message(row, img_dir)
    answer = ast.literal_eval(row['Answer'])
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target_text}]})
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False


# 학습: qwen3vl_final 레시피 동결 + GT-인접 고-cosine 오버샘플링

- 레시피는 EXP-014(qwen3-8b-lora.ipynb)와 동일: LoRA r=16/α=32/dropout 0.05,
  attn q/k/v/o_proj, AdamW8bit lr=2e-4, batch 1 × grad accum 8, MAX_EPOCHS 5,
  patience 2, seed 42. **유일한 변경은 오버샘플 ×2 복제뿐이다.**
- 아래 두 셀(모델 로드 → dataset/collate)을 실행한 뒤 학습 셀을 실행한다.
- 학습 셀은 epoch 1의 optimizer step 20에서 `[PROBE]` 예상 소요를 출력한다.
  **세션 예산을 초과하면 그 시점에 런타임을 중단**하고 보고할 것 (epoch별
  adapter가 Drive에 남으므로 재개 가능).
- 세션이 끊기면 노트북을 처음부터 다시 실행 — 마지막 완결 epoch adapter에서
  자동 재개한다 (optimizer 모멘트는 초기화되는 원본 복구 흐름과 같은 한계).
- 저장 경로는 전부 `qwen3vl_v2_os_*`이며, 저장 함수가 다른 이름(기존 제출
  adapter 등)에 쓰는 것을 거부한다.


In [10]:
import gc
for variable_name in (
    'optimizer', 'outputs', 'raw_loss', 'loss', 'batch', 'model', 'base_model'
):
    globals().pop(variable_name, None)
gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임을 GPU로 설정한 뒤 다시 실행하세요.')
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'학습 GPU: {gpu_name} ({gpu_vram_gb:.1f} GiB)')
assert gpu_vram_gb >= 38, (
    f'VRAM {gpu_vram_gb:.1f} GiB — 픽셀 캡 없는 이 레시피는 A100 40GB급 이상에서만 검증되었습니다.'
)

processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map="auto",
)

def find_latest_completed_adapter(output_dir):
    candidates = []
    for path in Path(output_dir).glob(f'{ARM_NAME}_epoch*'):
        suffix = path.name.removeprefix(f'{ARM_NAME}_epoch')
        if not suffix.isdigit():
            continue
        has_config = (path / 'adapter_config.json').is_file()
        has_weights = (path / 'adapter_model.safetensors').is_file()
        if has_config and has_weights:
            candidates.append((int(suffix), path))
    if not candidates:
        return None, 0
    completed_epochs, path = max(candidates)
    return path, completed_epochs

RESUME_ADAPTER_DIR, COMPLETED_EPOCHS = find_latest_completed_adapter(OUTPUT_DIR)
if RESUME_ADAPTER_DIR is not None:
    model = PeftModel.from_pretrained(model, str(RESUME_ADAPTER_DIR), is_trainable=True)
    print(f'완료된 {COMPLETED_EPOCHS} epoch adapter에서 재개: {RESUME_ADAPTER_DIR}')
else:
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable_params == 15_335_424, trainable_params


학습 GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Error while downloading from https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct/resolve/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b/model-00001-of-00004.safetensors: The read operation timed out
Trying to resume download...
Trying to resume download...


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 15,335,424 || all params: 8,782,459,120 || trainable%: 0.1746


In [11]:
class QwenOrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(messages)

        full_enc = self.processor(text=[full_text], images=image_inputs, return_tensors="pt", padding=False)
        prompt_enc = self.processor(text=[prompt_only], images=image_inputs, return_tensors="pt", padding=False)

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[:prompt_len] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
            'mm_token_type_ids': full_enc['mm_token_type_ids'][0],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels, mm_token_type_ids = [], [], [], []
    for item in batch:
        seq_len = item['input_ids'].shape[0]
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        mm_token_type_ids.append(torch.cat([item['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    pixel_values = torch.cat([item['pixel_values'] for item in batch], dim=0)
    image_grid_thw = torch.cat([item['image_grid_thw'] for item in batch], dim=0)

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw,
        'mm_token_type_ids': torch.stack(mm_token_type_ids),
    }


In [12]:
import shutil
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다.')
torch.cuda.empty_cache()

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

def safe_save_adapter(model, path):
    # 이 팔의 저장 경로만 허용 — 기존 제출 adapter 디렉토리 보호.
    assert Path(path).name.startswith(ARM_NAME), f'쓰기 거부: {path}'
    tmp_path = path + '_tmp'
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    model.save_pretrained(tmp_path, selected_adapters=['default'])
    if os.path.exists(path):
        shutil.rmtree(path)
    os.rename(tmp_path, path)
    print(f"저장 완료: {path}")

fold = 0
tr_df = train[train['fold'] != fold].reset_index(drop=True)
va_df = train[train['fold'] == fold].reset_index(drop=True)
assert len(tr_df) == 7620 and len(va_df) == 1915, (len(tr_df), len(va_df))

# 유일한 레시피 변경: GT-인접 max cosine > 0.9 샘플을 한 번 더 넣는다 (x2).
dup_df = tr_df[tr_df['Id'].isin(OVERSAMPLE_IDS)].reset_index(drop=True)
assert len(dup_df) == len(OVERSAMPLE_IDS), (len(dup_df), len(OVERSAMPLE_IDS))
tr_df_os = pd.concat([tr_df, dup_df], ignore_index=True)
print(f"Train base: {len(tr_df)}, oversample duplicates: {len(dup_df)}, "
      f"total: {len(tr_df_os)}, Val: {len(va_df)}")

tr_ds = QwenOrderingDataset(tr_df_os, f'{DATA_DIR}/train', processor)
va_ds = QwenOrderingDataset(va_df, f'{DATA_DIR}/train', processor)
tr_loader = DataLoader(tr_ds, batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)
va_loader = DataLoader(va_ds, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2)

import bitsandbytes as bnb
optimizer = bnb.optim.AdamW8bit(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
GRAD_ACCUM_STEPS = 8
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
MAX_GRAD_NORM = 1.0
CHECKPOINT_EVERY = 500
PROBE_OPTIMIZER_STEPS = 20
BEST_ADAPTER_DIR = f'{OUTPUT_DIR}/{ARM_NAME}_best'
FINAL_ADAPTER_DIR = f'{OUTPUT_DIR}/{ARM_NAME}_final'
TRAINING_LOG_PATH = Path(OUTPUT_DIR) / f'{ARM_NAME}_training_log.json'

def evaluate_validation_loss(model, loader):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    val_started = time.perf_counter()
    with torch.inference_mode():
        for batch_idx, batch in enumerate(tqdm(loader, desc='Validation loss', leave=False)):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            val_loss = model(**batch).loss
            if not torch.isfinite(val_loss):
                raise RuntimeError(f'비정상 validation loss: {val_loss.item()}')
            batch_size = batch['input_ids'].shape[0]
            loss_sum += val_loss.item() * batch_size
            sample_count += batch_size
            if batch_idx + 1 == 100:
                rate = (time.perf_counter() - val_started) / 100
                print(f'[PROBE] validation {rate:.2f} s/batch '
                      f'-> full pass ~{rate * len(loader) / 60:.1f} min', flush=True)
    return loss_sum / sample_count

# 이전 세션의 epoch 기록이 있으면 best/patience 상태를 복원한다. epoch adapter는
# validation 전에 저장되므로, validation 도중 끊긴 세션은 기록이 1개 모자란다 —
# 그 epoch의 validation만 다시 돌려 기록을 맞춘다 (원본 복구 흐름의 자동화).
best_val_loss = float('inf')
best_epoch = None
epochs_without_improvement = 0
training_history = []
recovered_this_session = False
if COMPLETED_EPOCHS > 0:
    if TRAINING_LOG_PATH.is_file():
        training_history = json.loads(TRAINING_LOG_PATH.read_text())['history']
    if len(training_history) == COMPLETED_EPOCHS - 1:
        print(f'epoch {COMPLETED_EPOCHS} validation 중 중단된 세션 — validation 재실행')
        recovered_val_loss = evaluate_validation_loss(model, va_loader)
        training_history.append({
            'epoch': COMPLETED_EPOCHS,
            'train_loss': None,
            'val_loss': recovered_val_loss,
            'recovered': True,
        })
        recovered_this_session = True
    assert len(training_history) == COMPLETED_EPOCHS, (len(training_history), COMPLETED_EPOCHS)
    for entry in training_history:
        if entry['val_loss'] < best_val_loss - EARLY_STOPPING_MIN_DELTA:
            best_val_loss = entry['val_loss']
            best_epoch = entry['epoch']
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
    print(f'훈련 로그 복원: {len(training_history)} epochs, '
          f'best epoch={best_epoch} (val_loss={best_val_loss:.4f})')
    # 복구된 epoch가 best인데 세션이 best 저장 전에 끊겼던 경우를 메운다.
    if recovered_this_session and best_epoch == COMPLETED_EPOCHS:
        safe_save_adapter(model, BEST_ADAPTER_DIR)
        safe_save_adapter(model, FINAL_ADAPTER_DIR)
        print(f'복구 epoch {best_epoch}가 best — best/final adapter 재저장')

def write_training_log(status):
    payload = {
        'arm': ARM_NAME,
        'status': status,
        'gpu': gpu_name,
        'config': {
            'model': MODEL_NAME, 'revision': MODEL_REVISION,
            'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05,
                     'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj']},
            'lr': 2e-4, 'batch_size': 1, 'grad_accum_steps': GRAD_ACCUM_STEPS,
            'max_epochs': MAX_EPOCHS, 'patience': EARLY_STOPPING_PATIENCE,
            'min_delta': EARLY_STOPPING_MIN_DELTA, 'seed': SEED,
            'n_train_base': len(tr_df), 'n_oversample_dup': len(dup_df),
            'n_train_total': len(tr_df_os), 'n_val': len(va_df),
        },
        'history': training_history,
        'best_epoch': best_epoch,
        'best_val_loss': None if best_val_loss == float('inf') else best_val_loss,
    }
    TRAINING_LOG_PATH.write_text(json.dumps(payload, indent=2) + '\n')

for epoch in range(COMPLETED_EPOCHS, MAX_EPOCHS):
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'이미 early stopping 조건 충족 — epoch {epoch + 1} 건너뜀 (best={best_epoch})')
        break
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    epoch_started = time.perf_counter()
    optimizer_steps = 0
    for step, batch in enumerate(tqdm(tr_loader, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}")):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        raw_loss = outputs.loss
        if not torch.isfinite(raw_loss):
            raise RuntimeError(f'비정상 train loss: step={step}, loss={raw_loss.item()}')
        (raw_loss / GRAD_ACCUM_STEPS).backward()

        is_update_step = ((step + 1) % GRAD_ACCUM_STEPS == 0) or (step + 1 == len(tr_loader))
        if is_update_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            optimizer_steps += 1
            if epoch == COMPLETED_EPOCHS and optimizer_steps == PROBE_OPTIMIZER_STEPS:
                elapsed = time.perf_counter() - epoch_started
                sec_per_opt_step = elapsed / optimizer_steps
                steps_per_epoch = math.ceil(len(tr_loader) / GRAD_ACCUM_STEPS)
                epoch_hours = sec_per_opt_step * steps_per_epoch / 3600
                print(
                    f"\n[PROBE] {optimizer_steps} optimizer steps in {elapsed:.0f}s "
                    f"({sec_per_opt_step:.1f} s/step) -> ~{epoch_hours:.2f} h/epoch, "
                    f"최악 {MAX_EPOCHS} epochs ~{epoch_hours * MAX_EPOCHS:.1f} h (+validation). "
                    f"세션 예산 초과가 예상되면 지금 중단하세요.",
                    flush=True,
                )

        total_loss += raw_loss.item()
        if step % 100 == 0:
            print(f" step {step}: loss={raw_loss.item():.4f}")

        if (step + 1) % CHECKPOINT_EVERY == 0:
            safe_save_adapter(model, f'{OUTPUT_DIR}/{ARM_NAME}_checkpoint_latest')

    train_loss = total_loss / len(tr_loader)
    # validation 중 런타임이 끊겨도 학습 완료 adapter는 Drive에 남긴다.
    safe_save_adapter(model, f'{OUTPUT_DIR}/{ARM_NAME}_epoch{epoch + 1}')
    val_loss = evaluate_validation_loss(model, va_loader)
    epoch_wall_seconds = time.perf_counter() - epoch_started
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'wall_seconds': round(epoch_wall_seconds, 1),
    })
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"wall={epoch_wall_seconds / 3600:.2f} h")

    if val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        safe_save_adapter(model, BEST_ADAPTER_DIR)
        safe_save_adapter(model, FINAL_ADAPTER_DIR)
        print(f"새 best adapter: epoch={best_epoch}, val_loss={best_val_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"개선 없음: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")
    write_training_log('in_progress')
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}; best epoch={best_epoch}")
        break

write_training_log('completed')
print(f"Best epoch={best_epoch}, best validation loss={best_val_loss:.4f}")
print(f"Best adapter path: {FINAL_ADAPTER_DIR}")


Train base: 7620, oversample duplicates: 3194, total: 10814, Val: 1915


Epoch 1/5:   0%|          | 0/10814 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


 step 0: loss=0.9120
 step 100: loss=0.1777

[PROBE] 20 optimizer steps in 87s (4.4 s/step) -> ~1.64 h/epoch, 최악 5 epochs ~8.2 h (+validation). 세션 예산 초과가 예상되면 지금 중단하세요.
 step 200: loss=0.0069
 step 300: loss=0.2152
 step 400: loss=0.1099
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 500: loss=0.0252
 step 600: loss=0.3004
 step 700: loss=0.1495
 step 800: loss=0.0063
 step 900: loss=0.0276
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 1000: loss=0.1693
 step 1100: loss=0.0218
 step 1200: loss=0.0655
 step 1300: loss=0.0375
 step 1400: loss=0.0745
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 1500: loss=0.2073
 step 1600: loss=0.1576
 step 1700: loss=0.1739
 step 1800: loss=0.1717
 step 1900: loss=0.1916
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 2000: loss=0.3131
 step 2100: loss=0.1017
 st

Validation loss:   0%|          | 0/1915 [00:00<?, ?it/s]

...

 step 800: loss=0.0036
 step 900: loss=0.0000
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 1000: loss=0.0003
 step 1100: loss=0.0000
 step 1200: loss=0.3055
 step 1300: loss=0.0002
 step 1400: loss=0.0001
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 1500: loss=0.2670
 step 1600: loss=0.1160
 step 1700: loss=0.0867
 step 1800: loss=0.0959
 step 1900: loss=0.0000
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 2000: loss=0.0002
 step 2100: loss=0.0337
 step 2200: loss=0.0375
 step 2300: loss=0.0031
 step 2400: loss=0.0249
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 2500: loss=0.0000
 step 2600: loss=0.0690
 step 2700: loss=0.0009
 step 2800: loss=0.1409
 step 2900: loss=0.0000
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_checkpoint_latest
 step 3000: loss=0.0001
 step

Validation loss:   0%|          | 0/1915 [00:00<?, ?it/s]

[PROBE] validation 0.18 s/batch -> full pass ~5.7 min


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f6a1da73560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
     if w.is_alive():
      ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f6a1da73560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 3: train_loss=0.0616, val_loss=0.1244, wall=1.71 h
개선 없음: 2/2
Early stopping at epoch 3; best epoch=1
Best epoch=1, best validation loss=0.1163
Best adapter path: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_final


# 저장 검증 + 즉시 백업 (학습 직후 반드시 실행)

EXP-012 full-param 체크포인트는 Drive 저장이 조용히 잘려 shard 1-4가 유실됐다.
아래 셀은 Drive에 저장된 `qwen3vl_v2_os_*` 디렉토리 전부를 **다시 읽어**
파일 개수·크기·sha256 매니페스트를 만들고, final adapter를 zip으로 즉시
브라우저 다운로드한다(1차 백업, ~60 MB). 다운로드가 시작되는 것까지 확인한
뒤에 런타임을 종료할 것.


In [13]:
import zipfile
from datetime import datetime, timezone

final_dir = Path(FINAL_ADAPTER_DIR)
assert final_dir.is_dir(), final_dir
adapter_config = json.loads((final_dir / 'adapter_config.json').read_text())
assert adapter_config['r'] == 16, adapter_config['r']
assert adapter_config['lora_alpha'] == 32, adapter_config['lora_alpha']
assert sorted(adapter_config['target_modules']) == \
    ['k_proj', 'o_proj', 'q_proj', 'v_proj'], adapter_config['target_modules']
weights_path = final_dir / 'adapter_model.safetensors'
assert weights_path.is_file(), weights_path
assert weights_path.stat().st_size >= 40 * 1024 * 1024, weights_path.stat().st_size

manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'arm': ARM_NAME,
    'gpu': gpu_name,
    'dirs': {},
}
for adapter_dir in sorted(Path(OUTPUT_DIR).glob(f'{ARM_NAME}_*')):
    if not adapter_dir.is_dir():
        continue
    files_entry = {}
    for file in sorted(adapter_dir.rglob('*')):
        if file.is_file():
            data = file.read_bytes()  # Drive 재읽기 — silent truncation 검출
            files_entry[file.relative_to(adapter_dir).as_posix()] = {
                'bytes': len(data),
                'sha256': hashlib.sha256(data).hexdigest(),
            }
    assert files_entry, f'빈 디렉토리: {adapter_dir}'
    manifest['dirs'][adapter_dir.name] = files_entry

manifest_path = Path(OUTPUT_DIR) / f'{ARM_NAME}_save_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
print({name: len(entry) for name, entry in manifest['dirs'].items()})
print(f'manifest: {manifest_path}')

backup_zip = Path(f'/content/{ARM_NAME}_final_backup.zip')
if backup_zip.exists():
    backup_zip.unlink()
with zipfile.ZipFile(backup_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(final_dir.rglob('*')):
        if file.is_file():
            archive.write(file, f'{ARM_NAME}_final/{file.relative_to(final_dir).as_posix()}')
    archive.write(manifest_path, manifest_path.name)
    if TRAINING_LOG_PATH.is_file():
        archive.write(TRAINING_LOG_PATH, TRAINING_LOG_PATH.name)

from google.colab import files
files.download(str(backup_zip))


{'qwen3vl_v2_os_best': 3, 'qwen3vl_v2_os_checkpoint_latest': 3, 'qwen3vl_v2_os_epoch1': 3, 'qwen3vl_v2_os_epoch2': 3, 'qwen3vl_v2_os_epoch3': 3, 'qwen3vl_v2_os_final': 3}
manifest: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_save_manifest.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# fold-0 holdout 예측 → ft_val_preds.csv

학습이 끝난 뒤 (또는 새 런타임에서 위쪽 준비 셀들 실행 후) 아래 두 셀을
순서대로 실행한다.

1. base + `qwen3vl_v2_os_final` adapter 로드 셀
2. fold-0 1,915행 추론 셀 — `ft_val_preds.csv`(Id,pred,parse_ok,gen_time_ms,
   기존 `outputs/lora_holdout_infer/ft_val_preds.csv`와 동일 규격)와 상세 CSV를
   Drive `snu_qwen3vl_outputs/qwen3vl_v2_os_holdout/`에 저장하고 다운로드한다.

로컬 회수 후 `ft_val_preds.csv`는 `outputs/qwen3vl_v2_os_holdout_infer/`에
놓는다. test 데이터는 이 노트북 어디서도 사용하지 않는다.


In [14]:
import gc
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
FINAL_ADAPTER_DIR = Path(OUTPUT_DIR) / f'{ARM_NAME}_final'
assert (FINAL_ADAPTER_DIR / 'adapter_config.json').is_file(), FINAL_ADAPTER_DIR
assert (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file(), FINAL_ADAPTER_DIR

for variable_name in ('model', 'base_model'):
    globals().pop(variable_name, None)
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR))
model.eval()
print(f'Adapter loaded: {FINAL_ADAPTER_DIR}')

for name, param in model.named_parameters():
    if 'lora_B' in name:
        print(f"LoRA B 값 확인: {param.abs().mean().item()}")
        break


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Adapter loaded: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_final
LoRA B 값 확인: 0.00621941639110446


In [15]:
from scipy.stats import kendalltau

if 'fold' not in train.columns:
    raise RuntimeError('fold 로드 셀을 먼저 실행하세요.')
fold = 0
va_df = train[train['fold'] == fold].reset_index(drop=True)
print(f'Validation samples: {len(va_df)}')
assert len(va_df) == 1915, len(va_df)

def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [kendalltau(p,t)[0] if not np.isnan(kendalltau(p,t)[0]) else 1.0 for p,t in zip(pred,target)]
    return {'exact_match': exact_match, 'position_acc': position_acc, 'kendall_tau': np.mean(taus)}

processor.tokenizer.padding_side = 'left'  # 배치 생성을 위해 필수

BATCH_SIZE = 8

model.eval()
results = []

for start_idx in tqdm(range(0, len(va_df), BATCH_SIZE)):
    batch_rows = va_df.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, f'{DATA_DIR}/train') for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    batch_started = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    per_sample_ms = (time.perf_counter() - batch_started) * 1000.0 / len(batch_rows)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        results.append({
            'Id': row['Id'],
            'pred': pred,
            'parse_ok': parse_ok,
            'gen_time_ms': round(per_sample_ms, 1),
            'raw_output': output_text,
        })

assert len(results) == len(va_df), (len(results), len(va_df))
assert [r['Id'] for r in results] == va_df['Id'].tolist(), 'validation Id 순서가 달라졌습니다.'
assert all(sorted(r['pred']) == [1, 2, 3, 4] for r in results), '비정상 pred가 있습니다.'

pred_array = np.array([r['pred'] for r in results]) - 1
target_array = np.array([ast.literal_eval(a) for a in va_df['Answer']]) - 1

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df['No_ordering'].values
shuffled = compute_metrics(pred_array[~no_order_mask], target_array[~no_order_mask])

print(f"[{ARM_NAME}, Group-CV val fold 0, n={len(va_df)}]")
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={1-np.mean([r['parse_ok'] for r in results]):.1%}")

HOLDOUT_OUT_DIR = Path(OUTPUT_DIR) / f'{ARM_NAME}_holdout'
HOLDOUT_OUT_DIR.mkdir(parents=True, exist_ok=True)

# scripts/run_hybrid_judgment.py --ft-preds 가 그대로 읽는 규격.
ft_val_preds_df = pd.DataFrame({
    'Id': [r['Id'] for r in results],
    'pred': [str(r['pred']) for r in results],
    'parse_ok': [r['parse_ok'] for r in results],
    'gen_time_ms': [r['gen_time_ms'] for r in results],
})
ft_val_preds_path = HOLDOUT_OUT_DIR / 'ft_val_preds.csv'
ft_val_preds_df.to_csv(ft_val_preds_path, index=False)

validation_predictions_df = pd.DataFrame({
    'Id': va_df['Id'],
    'Fold': fold,
    'Answer': va_df['Answer'],
    'Prediction': [str(r['pred']) for r in results],
    'Correct': (pred_array == target_array).all(axis=1),
    'ParseOk': [r['parse_ok'] for r in results],
    'No_ordering': va_df['No_ordering'],
    'GenTimeMs': [r['gen_time_ms'] for r in results],
    'RawOutput': [r['raw_output'] for r in results],
})
validation_csv_path = HOLDOUT_OUT_DIR / f'validation_predictions_{ARM_NAME}_fold0.csv'
validation_predictions_df.to_csv(validation_csv_path, index=False, encoding='utf-8-sig')

saved_preds_df = pd.read_csv(ft_val_preds_path)
assert len(saved_preds_df) == 1915
assert saved_preds_df['Id'].tolist() == va_df['Id'].tolist()
print(f'저장 완료: {ft_val_preds_path}')
print(f'저장 완료: {validation_csv_path}')

from google.colab import files
files.download(str(ft_val_preds_path))
files.download(str(validation_csv_path))


Validation samples: 1915


  0%|          | 0/240 [00:00<?, ?it/s]

[qwen3vl_v2_os, Group-CV val fold 0, n=1915]
  전체 exact=0.5269
  셔플됨 exact=0.4795
  파싱 실패율=0.0%
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_holdout/ft_val_preds.csv
저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_v2_os_holdout/validation_predictions_qwen3vl_v2_os_fold0.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>